# RFM Analysis - Online Retail II

## Business Context
I built this notebook to turn transaction-level retail data into customer-level signals that a business team could actually use. Instead of asking only who bought, I wanted to measure how recently each customer bought, how often they came back, and how much value they generated.

## Why check requirements first
I like validating the environment before I touch the database or build features. It keeps the notebook reproducible and removes avoidable setup friction when someone else runs the work.

In [1]:
# Verify the main libraries first so I do not discover an environment problem halfway through the analysis.
required_libraries = ["pandas", "sqlite3"]

print("Requirements check:")
for library_name in required_libraries:
    # Import each dependency here so the notebook fails fast if the environment is incomplete.
    __import__(library_name)
    print(f"- {library_name}: available")

Requirements check:
- pandas: available
- sqlite3: available


## Why pull transactions with SQL first
I wanted `retail.db` to serve as the clean handoff point between notebooks. Reading directly from SQLite keeps the workflow explicit and shows that I can move comfortably between SQL and Python in the same analysis.

In [2]:
# Keep the imports explicit so it is obvious which tools I rely on in this feature-engineering step.
import sqlite3
import pandas as pd

# Use the local SQLite database created in the ingestion notebook as my clean transaction source.
database_path = "retail.db"

# Load the full transactions table with SQL so the handoff between storage and analysis stays transparent.
sqlite_connection = sqlite3.connect(database_path)
sql_query = "SELECT * FROM transactions"
df_transactions = pd.read_sql_query(sql_query, sqlite_connection)

# Close the connection right away because I only need the data in memory from this point forward.
sqlite_connection.close()

print(f"Loaded rows: {len(df_transactions):,}")
print(f"Columns: {list(df_transactions.columns)}")

Loaded rows: 407,695
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country']


## Why define a snapshot date and build RFM here
RFM works best when I anchor every customer against the same reference point in time. I use a snapshot date one day after the most recent invoice, then I aggregate behavior at the customer level so I can compare people instead of individual transactions.

In [3]:
# Copy the transactions first so I can engineer features without mutating the raw SQL pull.
df_transactions_working = df_transactions.copy()

# Validate the expected schema now because RFM breaks quietly if even one core field is missing.
required_columns = ["CustomerID", "Invoice", "InvoiceDate", "Quantity", "Price"]
missing_columns = [column for column in required_columns if column not in df_transactions_working.columns]
if missing_columns:
    raise KeyError(
        "Required columns are missing from transactions table: "
        f"{missing_columns}"
    )

# Convert InvoiceDate before any grouping because Recency depends on accurate date arithmetic.
df_transactions_working["InvoiceDate"] = pd.to_datetime(
    df_transactions_working["InvoiceDate"], errors="coerce"
)

# Drop invalid dates because one broken timestamp should not poison a customer-level summary.
df_transactions_working = df_transactions_working.dropna(subset=["InvoiceDate"])

# Coerce quantity and price to numeric so the monetary calculation does not fail on messy source values.
df_transactions_working["Quantity"] = pd.to_numeric(
    df_transactions_working["Quantity"], errors="coerce"
)
df_transactions_working["Price"] = pd.to_numeric(
    df_transactions_working["Price"], errors="coerce"
)

# Remove any rows that still cannot support revenue math after coercion.
df_transactions_working = df_transactions_working.dropna(subset=["Quantity", "Price"])

# Set the snapshot date one day after the latest invoice so even the most recent customer gets a Recency of at least one day.
snapshot_date = df_transactions_working["InvoiceDate"].max() + pd.Timedelta(days=1)
print(f"Snapshot date: {snapshot_date}")

# Build a line-level revenue field here because Monetary should reflect actual spend, not just order counts.
df_transactions_working["LineTotal"] = (
    df_transactions_working["Quantity"] * df_transactions_working["Price"]
)

# Aggregate to the customer level because that is the unit the clustering model will ultimately segment.
df_rfm = (
    df_transactions_working
    .groupby("CustomerID")
    .agg({
        "InvoiceDate": lambda invoice_dates: (snapshot_date - invoice_dates.max()).days,
        "Invoice": "nunique",
        "LineTotal": "sum"
    })
    .reset_index()
)

# Rename the outputs to clean RFM labels so the meaning is obvious in the next notebook.
df_rfm = df_rfm.rename(
    columns={
        "InvoiceDate": "Recency",
        "Invoice": "Frequency",
        "LineTotal": "Monetary"
    }
)

# Keep only positive-Monetary customers because zero or negative values would blur the customer-value story I am trying to tell.
df_rfm = df_rfm[df_rfm["Monetary"] > 0]

# Sort by CustomerID mainly to make inspection and debugging less chaotic when I revisit the table later.
df_rfm = df_rfm.sort_values(by="CustomerID").reset_index(drop=True)

print(f"RFM customers retained: {len(df_rfm):,}")

Snapshot date: 2010-12-10 20:01:00
RFM customers retained: 4,312


## Why preview and summarize the RFM table
This is the point where I check whether the feature engineering produced something believable. A quick preview plus summary statistics helps me spot unreasonable ranges before I feed the features into K-Means.

In [4]:
# Inspect the first 10 rows to make sure the customer-level metrics look sensible before modeling.
display(df_rfm.head(10))

# Use describe here to get a fast read on spread, outliers, and whether Monetary is as skewed as I expect.
display(df_rfm.describe(include="all"))

,CustomerID,Recency,Frequency,Monetary
0,12346.0,165,11,372.86
1,12347.0,3,2,1323.32
2,12348.0,74,1,222.16
3,12349.0,43,3,2671.14
4,12351.0,11,1,300.93
5,12352.0,11,2,343.80
6,12353.0,44,1,317.76
7,12355.0,203,1,488.21
8,12356.0,16,3,3562.25
9,12357.0,24,2,12079.99


,CustomerID,Recency,Frequency,Monetary
count,4312.000000,4312.000000,4312.000000,4312.000000
mean,15349.290353,91.171846,4.455705,2048.238236
std,1701.200176,96.860633,8.170213,8914.481280
min,12346.000000,1.000000,1.000000,2.950000
25%,13882.500000,18.000000,1.000000,307.987500
50%,15350.500000,53.000000,2.000000,706.020000
75%,16834.250000,136.000000,5.000000,1723.142500
max,18287.000000,374.000000,205.000000,349164.350000
